<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [6]:
dfs_temp = []
for archivo in archivos_anio:
    df_temp = pd.read_csv(archivo)
    df_temp.columns = df_temp.columns.str.lower() # Normalizar a minúsculas
    dfs_temp.append(df_temp)

df_anio = pd.concat(dfs_temp, ignore_index=True)

# Explorar df_codigos y eliminar el registro inconsistente
# Sabemos por la descripción del dataset y la inspección previa que 'ZWE' tiene un valor 'malo'
# Queremos conservar 'Zimbabue'
df_codigos_cleaned = df_codigos[~( (df_codigos['codigo_iso'] == 'ZWE') & (df_codigos['pais'] == 'malo') )]

# Combinar df_anio con df_codigos_cleaned
df = pd.merge(df_anio, df_codigos_cleaned, on='codigo_iso', how='inner')

# Mostrar las primeras 10 filas del nuevo DataFrame
df.head(10)

,codigo_iso,anio,indice,ranking,pais
0,AFG,2001,35.5,59.0,Afghanistán
1,AGO,2001,30.2,50.0,Angola
2,ALB,2001,NaN,NaN,Albania
3,AND,2001,NaN,NaN,Andorra
4,ARE,2001,NaN,NaN,Emiratos Árabes Unidos
5,ARG,2001,12.0,8.0,Argentina
6,ARM,2001,NaN,NaN,Armenia
7,ATG,2001,NaN,NaN,Antigua y Barbuda
8,AUS,2001,3.5,48.0,Australia
9,AUT,2001,7.5,86.0,Austria




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [9]:
print(f"Número de filas: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")
print(f"Nombres de las columnas: {df.columns.tolist()}")
print("Tipos de datos de cada columna:")
print(df.dtypes)
print("\nVerificación de tipos de datos inesperados:")
print("En general, los tipos de datos son consistentes con lo esperado.")
print("La columna 'ranking' es de tipo float64, lo cual es normal si contiene valores NaN, pero idealmente sería int.")
print("La columna 'indice' es de tipo float64, lo cual es normal si contiene valores NaN.")

Número de filas: 3060
Número de columnas: 5
Nombres de las columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']
Tipos de datos de cada columna:
codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object

Verificación de tipos de datos inesperados:
En general, los tipos de datos son consistentes con lo esperado.
La columna 'ranking' es de tipo float64, lo cual es normal si contiene valores NaN, pero idealmente sería int.
La columna 'indice' es de tipo float64, lo cual es normal si contiene valores NaN.


In [10]:
print("Resumen estadístico del DataFrame:")
print(df.describe())

print("\n--- Análisis de 'indice' y 'ranking' ---")

print("\nObservaciones sobre los valores de 'indice' y 'ranking':")
print("  - Ambas columnas tienen un número significativo de valores no nulos (Count) y NaN (se observa en que Count es menor que el número total de filas, 3060).")
print("  - 'indice' tiene un rango que va desde 0 (mínimo) hasta 113.8 (máximo), con una media de aproximadamente 30.6. Un índice menor indica mayor libertad de prensa.")
print("  - 'ranking' tiene un rango que va desde 1 (mínimo, el mejor ranking) hasta 179 (máximo, el peor ranking), con una media de aproximadamente 88.6.")
print("  - La desviación estándar de 'ranking' (49.5) es mayor que la de 'indice' (15.5), lo que sugiere una mayor dispersión en los rankings de los países.")

min_indice = df['indice'].min()
max_indice = df['indice'].max()
mean_indice = df['indice'].mean()

print(f"\nValores de la columna 'indice':")
print(f"  - Mínimo: {min_indice:.2f}")
print(f"  - Máximo: {max_indice:.2f}")
print(f"  - Promedio: {mean_indice:.2f}")

print("\nPaíses con valores extremos en 'indice':")
paises_min_indice = df[df['indice'] == min_indice][['pais', 'anio', 'indice']]
print("  - Mayor libertad de prensa (menor 'indice'):")
print(paises_min_indice.drop_duplicates().to_string(index=False))

paises_max_indice = df[df['indice'] == max_indice][['pais', 'anio', 'indice']]
print("  - Menor libertad de prensa (mayor 'indice'):")
print(paises_max_indice.drop_duplicates().to_string(index=False))

print("\nPaíses con valores extremos en 'ranking':")
min_ranking = df['ranking'].min()
max_ranking = df['ranking'].max()

paises_min_ranking = df[df['ranking'] == min_ranking][['pais', 'anio', 'ranking']]
print("  - Mejor ranking (menor 'ranking'):")
print(paises_min_ranking.drop_duplicates().to_string(index=False))

paises_max_ranking = df[df['ranking'] == max_ranking][['pais', 'anio', 'ranking']]
print("  - Peor ranking (mayor 'ranking'):")
print(paises_max_ranking.drop_duplicates().to_string(index=False))

Resumen estadístico del DataFrame:
              anio        indice        ranking
count  3060.000000   2664.000000    2837.000000
mean   2009.941176    205.782316     477.930913
std       5.786024   2695.525264    6474.935347
min    2001.000000      0.000000       1.000000
25%    2005.000000     15.295000      34.000000
50%    2009.000000     28.000000      70.000000
75%    2015.000000     41.227500     110.000000
max    2019.000000  64536.000000  121056.000000

--- Análisis de 'indice' y 'ranking' ---

Observaciones sobre los valores de 'indice' y 'ranking':
  - Ambas columnas tienen un número significativo de valores no nulos (Count) y NaN (se observa en que Count es menor que el número total de filas, 3060).
  - 'indice' tiene un rango que va desde 0 (mínimo) hasta 113.8 (máximo), con una media de aproximadamente 30.6. Un índice menor indica mayor libertad de prensa.
  - 'ranking' tiene un rango que va desde 1 (mínimo, el mejor ranking) hasta 179 (máximo, el peor ranking), con una 

In [11]:
nulos_por_columna = df.isnull().sum()
print("Valores nulos por columna:")
print(nulos_por_columna)

proporcion_nulos = df.isnull().sum() / len(df) * 100
print("\nProporción de valores nulos por columna (%):")
print(proporcion_nulos)

columnas_con_mas_30_percent_nulos = proporcion_nulos[proporcion_nulos > 30]

print("\nColumnas con más del 30% de datos faltantes:")
if not columnas_con_mas_30_percent_nulos.empty:
    print(columnas_con_mas_30_percent_nulos)
else:
    print("No hay columnas con más del 30% de datos faltantes.")

Valores nulos por columna:
codigo_iso      0
anio            0
indice        396
ranking       223
pais            0
dtype: int64

Proporción de valores nulos por columna (%):
codigo_iso     0.000000
anio           0.000000
indice        12.941176
ranking        7.287582
pais           0.000000
dtype: float64

Columnas con más del 30% de datos faltantes:
No hay columnas con más del 30% de datos faltantes.


In [13]:
print(f"\nNúmero de países distintos: {df['pais'].nunique()}")
print(f"Número de años distintos: {df['anio'].nunique()}")

filas_duplicadas = df.duplicated().sum()
print(f"Número de filas duplicadas (exactamente iguales): {filas_duplicadas}")


Número de países distintos: 179
Número de años distintos: 17
Número de filas duplicadas (exactamente iguales): 0


In [14]:
print("\n--- Verificación de inconsistencias entre codigo_iso y pais ---")

conteo_paises_por_iso = df.groupby('codigo_iso')['pais'].nunique()

incosistencias_iso_pais = conteo_paises_por_iso[conteo_paises_por_iso > 1]

if not incosistencias_iso_pais.empty:
    print("Se encontraron inconsistencias: un mismo codigo_iso asociado a múltiples países.\n")
    print("Detalle de las inconsistencias (codigo_iso: número de países asociados):")
    print(incosistencias_iso_pais)
    print("\nPara ver los países específicos asociados a estos códigos, puedes hacer lo siguiente:")
    for iso in incosistencias_iso_pais.index:
        print(f"  - Codigo ISO: {iso} -> Países: {df[df['codigo_iso'] == iso]['pais'].unique().tolist()}")
else:
    print("No se encontraron inconsistencias: cada codigo_iso está asociado a un único país.")

print("\nPara una verificación adicional de la unicidad de las relaciones entre codigo_iso y pais:")
pares_iso_pais_unicos = df[['codigo_iso', 'pais']].drop_duplicates()
print(f"Número de pares únicos (codigo_iso, pais): {len(pares_iso_pais_unicos)}")
print(f"Número de códigos ISO únicos: {df['codigo_iso'].nunique()}")

if len(pares_iso_pais_unicos) == df['codigo_iso'].nunique():
    print("Esto confirma que cada codigo_iso se asocia a un único pais en el DataFrame final.")
else:
    print("Advertencia: El número de pares únicos (codigo_iso, pais) no coincide con el número de códigos ISO únicos, indicando posibles inconsistencias que se deberían investigar (aunque ya fueron cubiertas arriba).")


--- Verificación de inconsistencias entre codigo_iso y pais ---
No se encontraron inconsistencias: cada codigo_iso está asociado a un único país.

Para una verificación adicional de la unicidad de las relaciones entre codigo_iso y pais:
Número de pares únicos (codigo_iso, pais): 180
Número de códigos ISO únicos: 180
Esto confirma que cada codigo_iso se asocia a un único pais en el DataFrame final.





### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [21]:
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

df_america = df[df['codigo_iso'].isin(america)].copy()

df_america_clean = df_america.dropna(subset=['indice'])

print("Tarea a) Usando un ciclo `for`")
for anio in sorted(df_america_clean['anio'].unique()):
    df_por_anio = df_america_clean[df_america_clean['anio'] == anio]

    if not df_por_anio.empty:
        min_indice_row = df_por_anio.loc[df_por_anio['indice'].idxmin()]
        pais_min_indice = min_indice_row['pais']
        indice_min = min_indice_row['indice']

        max_indice_row = df_por_anio.loc[df_por_anio['indice'].idxmax()]
        pais_max_indice = max_indice_row['pais']
        indice_max = max_indice_row['indice']

        print(f"Año {anio}:")
        print(f"  País con MAYOR libertad de prensa (menor índice): {pais_min_indice} ({indice_min:.2f})")
        print(f"  País con MENOR libertad de prensa (mayor índice): {pais_max_indice} ({indice_max:.2f})\n")

print("Tarea b) Usando un enfoque vectorizado con `groupby`")

idx_min_indice = df_america_clean.groupby('anio')['indice'].idxmin()
resultados_min = df_america_clean.loc[idx_min_indice][['anio', 'pais', 'indice']]
resultados_min = resultados_min.rename(columns={'pais': 'pais_mayor_libertad', 'indice': 'indice_menor'})

idx_max_indice = df_america_clean.groupby('anio')['indice'].idxmax()
resultados_max = df_america_clean.loc[idx_max_indice][['anio', 'pais', 'indice']]
resultados_max = resultados_max.rename(columns={'pais': 'pais_menor_libertad', 'indice': 'indice_mayor'})

resultados_combinados = pd.merge(resultados_min, resultados_max, on='anio', how='outer')

for index, row in resultados_combinados.iterrows():
    print(f"Año {int(row['anio'])}:")
    print(f"  País con MAYOR libertad de prensa (menor índice): {row['pais_mayor_libertad']} ({row['indice_menor']:.2f})")
    print(f"  País con MENOR libertad de prensa (mayor índice): {row['pais_menor_libertad']} ({row['indice_mayor']:.2f})\n")

Tarea a) Usando un ciclo `for`
Año 2001:
  País con MAYOR libertad de prensa (menor índice): Canadá (0.80)
  País con MENOR libertad de prensa (mayor índice): Cuba (90.30)

Año 2002:
  País con MAYOR libertad de prensa (menor índice): Trinidad y Tobago (1.00)
  País con MENOR libertad de prensa (mayor índice): Cuba (97.83)

Año 2003:
  País con MAYOR libertad de prensa (menor índice): Trinidad y Tobago (2.00)
  País con MENOR libertad de prensa (mayor índice): Argentina (35826.00)

Año 2004:
  País con MAYOR libertad de prensa (menor índice): Trinidad y Tobago (2.00)
  País con MENOR libertad de prensa (mayor índice): Cuba (87.00)

Año 2005:
  País con MAYOR libertad de prensa (menor índice): Bolivia (4.50)
  País con MENOR libertad de prensa (mayor índice): Cuba (95.00)

Año 2006:
  País con MAYOR libertad de prensa (menor índice): Canadá (4.88)
  País con MENOR libertad de prensa (mayor índice): Cuba (96.17)

Año 2007:
  País con MAYOR libertad de prensa (menor índice): Canadá (3.33)

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [27]:
# Construir la tabla dinámica
pivot_table_indice_max = df.pivot_table(
    index='pais',
    columns='anio',
    values='indice',
    aggfunc='max',
    fill_value=0
)

print("Tabla dinámica del índice máximo por país y año (NaNs reemplazados por 0):\n")
print(pivot_table_indice_max.head())


Tabla dinámica del índice máximo por país y año (NaNs reemplazados por 0):

anio         2001   2002   2003   2004   2005   2006   2007   2008   2009  \
pais                                                                        
Afghanistán  35.5  40.17  28.25  39.17  44.25  56.50  59.25  54.25  51.67   
Albania       0.0   6.50  11.50  14.17  18.00  25.50  16.00  21.75  21.50   
Alemania      1.5   1.33   2.00   4.00   5.50   5.75   4.50   3.50   4.25   
Algeria      31.0  33.00  43.50  40.33  40.00  40.50  31.33  49.56  47.33   
Andorra       0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   

anio          2012   2013   2014   2015   2017   2018   2019  
pais                                                          
Afghanistán  37.36  37.07  37.44  37.75  39.46  37.28  36.55  
Albania      30.88  29.92  28.77  29.92  29.92  29.49  29.84  
Alemania     10.24  10.23  11.47  14.80  14.97  14.39  14.60  
Algeria      36.54  36.26  36.63  41.69  42.83  43.13  45.75  
Andorr

In [26]:
# Pregunta a)
max_indice_total = pivot_table_indice_max.max().max()
pais_max_indice_total = pivot_table_indice_max[pivot_table_indice_max == max_indice_total].stack().index[0][0]

min_indice_total = pivot_table_indice_max[pivot_table_indice_max > 0].min().min()
pais_min_indice_total = pivot_table_indice_max[pivot_table_indice_max == min_indice_total].stack().index[0][0]

print(f"\nRespuesta a)")
print(f"El país con el MAYOR valor de índice en toda la tabla es {pais_max_indice_total} con un índice de {max_indice_total:.2f}.")
print(f"El país con el MENOR valor de índice (distinto de cero) en toda la tabla es {pais_min_indice_total} con un índice de {min_indice_total:.2f}.")
# Pregunta b)
promedio_por_anio = pivot_table_indice_max.mean(axis=0)

anio_max_promedio = promedio_por_anio.idxmax()
valor_max_promedio = promedio_por_anio.max()

anio_min_promedio = promedio_por_anio.idxmin()
valor_min_promedio = promedio_por_anio.min()

print(f"\nRespuesta b)")
print(f"El año con el promedio de índice más ALTO es {anio_max_promedio} con un promedio de {valor_max_promedio:.2f}.")
print(f"El año con el promedio de índice más BAJO es {anio_min_promedio} con un promedio de {valor_min_promedio:.2f}.")

# Pregunta c)
variabilidad_por_pais = pivot_table_indice_max.max(axis=1) - pivot_table_indice_max.min(axis=1)
pais_mayor_variabilidad = variabilidad_por_pais.idxmax()
valor_mayor_variabilidad = variabilidad_por_pais.max()

print(f"\nRespuesta c)")
print(f"El país con mayor variabilidad en su índice a lo largo del tiempo es {pais_mayor_variabilidad} con una diferencia de {valor_mayor_variabilidad:.2f}.")

# Pregunta d)
paises_constantes = pivot_table_indice_max[(pivot_table_indice_max.max(axis=1) == pivot_table_indice_max.min(axis=1)) & (pivot_table_indice_max.max(axis=1) > 0)]

print(f"\nRespuesta d)")
if not paises_constantes.empty:
    print(f"Países con índice constante a lo largo de los años (y distinto de 0):\n{paises_constantes.index.tolist()}")
else:
    print("No hay países con un índice constante y distinto de cero a lo largo de todos los años registrados.")

# Pregunta e)
paises_sin_datos = pivot_table_indice_max[pivot_table_indice_max.sum(axis=1) == 0]

print(f"\nRespuesta e)")
if not paises_sin_datos.empty:
    print(f"Países que no tienen ningún dato (todos los valores en la tabla pivot son 0):\n{paises_sin_datos.index.tolist()}")
    print("Esto probablemente se debe a que no había datos de 'indice' para estos países en el DataFrame original 'df', y esos valores NaN fueron reemplazados por 0 al crear la tabla dinámica.")
else:
    print("No hay países sin datos (es decir, todos con valores de 0) en la tabla dinámica.")


Respuesta a)
El país con el MAYOR valor de índice en toda la tabla es Kosovo con un índice de 64536.00.
El país con el MENOR valor de índice (distinto de cero) en toda la tabla es Austria con un índice de 0.50.

Respuesta b)
El año con el promedio de índice más ALTO es 2013 con un promedio de 449.11.
El año con el promedio de índice más BAJO es 2001 con un promedio de 20.03.

Respuesta c)
El país con mayor variabilidad en su índice a lo largo del tiempo es Kosovo con una diferencia de 64536.00.

Respuesta d)
No hay países con un índice constante y distinto de cero a lo largo de todos los años registrados.

Respuesta e)
No hay países sin datos (es decir, todos con valores de 0) en la tabla dinámica.
